# Diffusion Foundations Exam

**Day 4 Assessment — Notebooks 07-09, with review of 00-06**

---

## Rules

- **Total time: 2 hours** (15 min warmup + 90 min main + 15 min buffer)
- **No documentation, no notebooks, no searching.** Close everything except this file and a blank Python environment.
- **Honor system.** The point is to find your gaps, not to pass. Cheating only cheats yourself.
- Write clean, typed, documented code — as if submitting for code review.
- Run all test/assertion cells to verify your work. Do not modify test cells.
- If you get stuck on a problem for more than 5 minutes without progress, write pseudocode for partial credit and move on.

## Structure

| Section | Problems | Time Each | Total |
|---------|----------|-----------|-------|
| Warmup  | 3 quick recall | 5 min | 15 min |
| Main    | 3 implementation | 30 min | 90 min |

**Start your timer now.**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

---

# Warmup (15 minutes total)

Three quick recall problems. 5 minutes each. These should be automatic if you internalized the material.

## Warmup 1: Sinusoidal Timestep Embedding (5 min)

Implement positional encoding for diffusion timesteps.

**Interface:**
```python
def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
```

**Requirements:**
- `t` is shape `(B,)` — batch of integer timesteps
- Output is shape `(B, dim)` — embedding vectors
- First half of `dim` uses `sin`, second half uses `cos`
- Frequencies: `exp(-i * log(10000) / (half_dim - 1))` for `i` in `range(half_dim)`
- Formula: `sin(t * freq)` and `cos(t * freq)`

In [ ]:
# YOUR CODE HERE

def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    """Sinusoidal positional embedding for scalar timesteps.

    Args:
        t: Integer timesteps of shape (B,).
        dim: Embedding dimension (must be even).

    Returns:
        Embedding tensor of shape (B, dim). First half sin, second half cos.
    """
    raise NotImplementedError

In [ ]:
# === TESTS — DO NOT MODIFY ===

# Shape test
t_test = torch.tensor([0, 50, 100, 999])
emb = sinusoidal_embedding(t_test, dim=128)
assert emb.shape == (4, 128), f"Expected (4, 128), got {emb.shape}"

# Different timesteps must produce different embeddings
emb_0 = sinusoidal_embedding(torch.tensor([0]), dim=64)
emb_1 = sinusoidal_embedding(torch.tensor([1]), dim=64)
emb_500 = sinusoidal_embedding(torch.tensor([500]), dim=64)
assert not torch.allclose(emb_0, emb_1), "t=0 and t=1 should produce different embeddings"
assert not torch.allclose(emb_0, emb_500), "t=0 and t=500 should produce different embeddings"

# Same timestep must produce same embedding
emb_a = sinusoidal_embedding(torch.tensor([42]), dim=64)
emb_b = sinusoidal_embedding(torch.tensor([42]), dim=64)
assert torch.allclose(emb_a, emb_b), "Same timestep should produce identical embeddings"

# Batch dimension preserved
batch_emb = sinusoidal_embedding(torch.arange(16), dim=256)
assert batch_emb.shape == (16, 256), f"Expected (16, 256), got {batch_emb.shape}"

print("Warmup 1 passed.")

## Warmup 2: Cumulative Alpha Schedule (5 min)

Compute the cumulative product of signal retention from a beta noise schedule.

**Interface:**
```python
def compute_alpha_bar(betas: torch.Tensor) -> torch.Tensor:
```

**Requirements:**
- `betas` is shape `(T,)` — noise schedule
- `alpha_bar[t] = prod_{s=0}^{t} (1 - betas[s])`
- Output shape must match `betas` shape

In [ ]:
# YOUR CODE HERE

def compute_alpha_bar(betas: torch.Tensor) -> torch.Tensor:
    """Compute cumulative product of (1 - betas).

    Args:
        betas: Noise schedule of shape (T,).

    Returns:
        alpha_bar of shape (T,), where alpha_bar[t] = prod_{s=0}^{t} (1 - betas[s]).
    """
    raise NotImplementedError

In [ ]:
# === TESTS — DO NOT MODIFY ===

betas = torch.linspace(1e-4, 0.02, 1000)
alpha_bar = compute_alpha_bar(betas)

# Shape must match
assert alpha_bar.shape == betas.shape, f"Expected {betas.shape}, got {alpha_bar.shape}"

# Values must monotonically decrease
diffs = alpha_bar[1:] - alpha_bar[:-1]
assert (diffs <= 0).all(), "alpha_bar must be monotonically decreasing"

# First value should be close to 1 (very little noise at t=0)
assert abs(alpha_bar[0].item() - (1 - betas[0].item())) < 1e-5, (
    f"alpha_bar[0] should be ~{1 - betas[0].item():.6f}, got {alpha_bar[0].item():.6f}"
)

# Last value should be close to 0 (almost pure noise at t=T-1)
assert alpha_bar[-1].item() < 0.05, f"alpha_bar[-1] should be near 0, got {alpha_bar[-1].item():.4f}"

print("Warmup 2 passed.")

## Warmup 3: Gather and Reshape for Broadcasting (5 min)

Given a 1D schedule tensor of shape `(T,)` and a batch of timestep indices `t` of shape `(B,)`, gather the values at those timesteps and reshape for broadcasting with `(B, C, H, W)` image tensors.

**Interface:**
```python
def gather_and_reshape(schedule: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
```

**Requirements:**
- `schedule` is shape `(T,)`
- `t` is shape `(B,)` with integer indices into `schedule`
- Output shape: `(B, 1, 1, 1)` — ready to multiply with `(B, C, H, W)` tensors

In [ ]:
# YOUR CODE HERE

def gather_and_reshape(schedule: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """Gather schedule values at timestep indices and reshape for image broadcasting.

    Args:
        schedule: 1D tensor of shape (T,).
        t: Integer timestep indices of shape (B,).

    Returns:
        Gathered values reshaped to (B, 1, 1, 1).
    """
    raise NotImplementedError

In [ ]:
# === TESTS — DO NOT MODIFY ===

schedule = torch.linspace(0.0, 1.0, 1000)
t = torch.tensor([0, 100, 500, 999])

result = gather_and_reshape(schedule, t)

# Shape must be (B, 1, 1, 1)
assert result.shape == (4, 1, 1, 1), f"Expected (4, 1, 1, 1), got {result.shape}"

# Values must match schedule at those indices
for i, idx in enumerate(t):
    expected = schedule[idx].item()
    actual = result[i, 0, 0, 0].item()
    assert abs(actual - expected) < 1e-5, f"At t={idx}: expected {expected:.6f}, got {actual:.6f}"

# Broadcasting test: must multiply cleanly with (B, C, H, W)
images = torch.randn(4, 3, 32, 32)
scaled = result * images
assert scaled.shape == (4, 3, 32, 32), f"Broadcasting failed: got {scaled.shape}"

print("Warmup 3 passed.")

---

# Main Problems (90 minutes total)

Three substantial implementation problems. 30 minutes each. These test deep understanding.

## Problem 1: Multi-Head Self-Attention from Scratch (30 min)

Implement multi-head self-attention as used in transformer-based diffusion models (DiT, U-ViT).

**Interface:**
```python
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int) -> None: ...
    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor: ...
```

**Requirements:**
- Input `x` is shape `(B, T, D)`, output is shape `(B, T, D)`
- `d_model` must be divisible by `n_heads`
- Compute `head_dim = d_model // n_heads`
- **Q/K/V projections:** Three separate `nn.Linear(d_model, d_model)` layers
- **Reshape to heads:** `(B, T, D)` -> `(B, T, n_heads, head_dim)` -> `(B, n_heads, T, head_dim)`
- **Scaled dot-product attention:** `scores = (Q @ K^T) / sqrt(head_dim)`, then softmax over last dim
- **Optional causal mask:** If `mask` is provided (shape `(1, 1, T, T)` or `(B, 1, T, T)`), apply `masked_fill` with `-inf` where `mask == 0` before softmax
- **Concat heads:** `(B, n_heads, T, head_dim)` -> `(B, T, D)`
- **Output projection:** Final `nn.Linear(d_model, d_model)`
- **No `nn.MultiheadAttention`.** Only use `nn.Linear`, `nn.Module`, basic tensor ops, `F.softmax`.

In [ ]:
# YOUR CODE HERE

class MultiHeadAttention(nn.Module):
    """Multi-head self-attention from scratch."""

    def __init__(self, d_model: int, n_heads: int) -> None:
        raise NotImplementedError

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        """Compute multi-head self-attention.

        Args:
            x: Input tensor of shape (B, T, D).
            mask: Optional mask of shape (1, 1, T, T) or (B, 1, T, T).
                  Positions where mask == 0 are masked out.

        Returns:
            Output tensor of shape (B, T, D).
        """
        raise NotImplementedError

In [ ]:
# === TESTS — DO NOT MODIFY ===

torch.manual_seed(42)

# --- Test 1: Output shape ---
mha = MultiHeadAttention(d_model=64, n_heads=8)
x = torch.randn(2, 10, 64)
out = mha(x)
assert out.shape == (2, 10, 64), f"Expected (2, 10, 64), got {out.shape}"

# --- Test 2: Causal mask prevents attending to future ---
mha2 = MultiHeadAttention(d_model=32, n_heads=4)
mha2.eval()

# Create two sequences: one length 5, one length 8 (padded)
# The first token's output should be identical regardless of future tokens
x_short = torch.randn(1, 5, 32)
x_long = torch.cat([x_short, torch.randn(1, 3, 32)], dim=1)  # (1, 8, 32)

causal_short = torch.tril(torch.ones(1, 1, 5, 5))
causal_long = torch.tril(torch.ones(1, 1, 8, 8))

with torch.no_grad():
    out_short = mha2(x_short, mask=causal_short)
    out_long = mha2(x_long, mask=causal_long)

# First token output should be identical (it can only attend to itself)
assert torch.allclose(out_short[0, 0, :], out_long[0, 0, :], atol=1e-5), (
    "Causal mask broken: first token output differs when future tokens are added"
)

# --- Test 3: n_heads=1 should behave like naive single-head attention ---
mha_single = MultiHeadAttention(d_model=16, n_heads=1)
x_small = torch.randn(1, 4, 16)
out_single = mha_single(x_small)
assert out_single.shape == (1, 4, 16), f"Single-head shape wrong: {out_single.shape}"

# Backward pass must work
loss = out.sum()
loss.backward()

print(f"Param count: {sum(p.numel() for p in mha.parameters()):,}")
print("Problem 1 passed.")

## Problem 2: Full DDPM — Forward Process, Training Step, Sampling (30 min)

Implement the three core DDPM operations given a pre-computed `alpha_bar` schedule.

### Part A: Forward Process
```python
def q_sample(x0: torch.Tensor, t: torch.Tensor, alpha_bar: torch.Tensor, noise: torch.Tensor | None = None) -> torch.Tensor:
```
- Reparameterization trick: `x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * noise`
- Must handle gathering from `alpha_bar` and reshaping for `(B, C, H, W)` broadcasting

### Part B: Training Step
```python
def ddpm_training_step(model: nn.Module, x0: torch.Tensor, optimizer: torch.optim.Optimizer, alpha_bar: torch.Tensor) -> float:
```
- Sample random `t` uniformly from `[0, T)`
- Sample noise, compute `x_t`, predict noise with model
- MSE loss between predicted and actual noise
- `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`
- Return loss as a Python float
- Model interface: `model(x_t, t)` returns predicted noise

### Part C: Sampling (Reverse Process)
```python
def ddpm_sample(model: nn.Module, alpha_bar: torch.Tensor, n_samples: int, img_shape: tuple, device: torch.device) -> torch.Tensor:
```
- Start from pure Gaussian noise
- Loop from `t = T-1` down to `t = 0`
- At each step: predict noise, compute posterior mean, add noise if `t > 0`
- Posterior mean: `mu = (1/sqrt(alpha_t)) * (x_t - (beta_t / sqrt(1 - alpha_bar_t)) * eps_pred)`
- Where `beta_t = 1 - alpha_t` and `alpha_t = alpha_bar[t] / alpha_bar[t-1]` (for t > 0, else `alpha_t = alpha_bar[0]`)
- Noise variance: `sqrt(beta_t) * z` where `z ~ N(0, I)`
- Return final denoised images `(n_samples, *img_shape)`

In [ ]:
# YOUR CODE HERE

def q_sample(
    x0: torch.Tensor,
    t: torch.Tensor,
    alpha_bar: torch.Tensor,
    noise: torch.Tensor | None = None,
) -> torch.Tensor:
    """DDPM forward process: noise clean images at timesteps t.

    Args:
        x0: Clean images (B, C, H, W).
        t: Timestep indices (B,).
        alpha_bar: Cumulative signal retention schedule (T,).
        noise: Optional pre-sampled noise, same shape as x0.

    Returns:
        Noisy images x_t of shape (B, C, H, W).
    """
    raise NotImplementedError


def ddpm_training_step(
    model: nn.Module,
    x0: torch.Tensor,
    optimizer: torch.optim.Optimizer,
    alpha_bar: torch.Tensor,
) -> float:
    """Single DDPM training step.

    Args:
        model: Noise prediction network, callable as model(x_t, t).
        x0: Batch of clean images (B, C, H, W).
        optimizer: Optimizer for model parameters.
        alpha_bar: Cumulative signal retention schedule (T,).

    Returns:
        Loss value as a Python float.
    """
    raise NotImplementedError


@torch.no_grad()
def ddpm_sample(
    model: nn.Module,
    alpha_bar: torch.Tensor,
    n_samples: int,
    img_shape: tuple,
    device: torch.device,
) -> torch.Tensor:
    """DDPM reverse process: generate images from noise.

    Args:
        model: Trained noise prediction network.
        alpha_bar: Cumulative signal retention schedule (T,).
        n_samples: Number of images to generate.
        img_shape: Shape of a single image (C, H, W).
        device: Torch device.

    Returns:
        Generated images of shape (n_samples, *img_shape).
    """
    raise NotImplementedError

In [ ]:
# === TESTS — DO NOT MODIFY ===

torch.manual_seed(123)
device = torch.device("cpu")

# Setup: schedule and dummy model
T = 200
betas = torch.linspace(1e-4, 0.02, T)
alpha_bar = torch.cumprod(1.0 - betas, dim=0)


class DummyNoiseModel(nn.Module):
    """Minimal model for testing: predicts noise via a single conv layer."""
    def __init__(self):
        super().__init__()
        self.net = nn.Conv2d(3, 3, 3, padding=1)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        return self.net(x)


model = DummyNoiseModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# --- Test q_sample: t=0 should be close to x0 ---
x0 = torch.randn(4, 3, 16, 16)
t_zero = torch.zeros(4, dtype=torch.long)
x_noised_t0 = q_sample(x0, t_zero, alpha_bar)
assert x_noised_t0.shape == x0.shape, f"q_sample shape: {x_noised_t0.shape}"
# At t=0, alpha_bar is ~1.0, so x_t should be very close to x0
residual_t0 = (x_noised_t0 - x0).abs().mean().item()
assert residual_t0 < 0.1, f"At t=0, x_t should be close to x0. Mean abs diff: {residual_t0:.4f}"

# --- Test q_sample: t=T-1 should be close to pure noise ---
t_last = torch.full((4,), T - 1, dtype=torch.long)
noise = torch.randn_like(x0)
x_noised_tmax = q_sample(x0, t_last, alpha_bar, noise=noise)
# At t=T-1, alpha_bar is ~0, so x_t should be close to pure noise
residual_noise = (x_noised_tmax - noise).abs().mean().item()
assert residual_noise < 0.15, f"At t=T-1, x_t should be close to noise. Mean abs diff: {residual_noise:.4f}"

# --- Test training step returns a float ---
loss_val = ddpm_training_step(model, x0, optimizer, alpha_bar)
assert isinstance(loss_val, float), f"Training step should return float, got {type(loss_val)}"
assert loss_val > 0, f"Loss should be positive, got {loss_val}"

# --- Test sampling output shape ---
model.eval()
samples = ddpm_sample(model, alpha_bar, n_samples=2, img_shape=(3, 16, 16), device=device)
assert samples.shape == (2, 3, 16, 16), f"Sample shape: expected (2, 3, 16, 16), got {samples.shape}"

print(f"q_sample t=0 residual: {residual_t0:.4f}")
print(f"q_sample t={T-1} noise residual: {residual_noise:.4f}")
print(f"Training loss: {loss_val:.4f}")
print(f"Sample shape: {samples.shape}")
print("Problem 2 passed.")

## Problem 3: Embedding Similarity Search (30 min)

*Review from earlier notebooks (data pipeline and deduplication).* Build a vectorized embedding search engine.

**Interface:**
```python
class EmbeddingSearcher:
    def __init__(self, embeddings: torch.Tensor, ids: list[str]) -> None: ...
    def search(self, query: torch.Tensor, top_k: int = 5) -> list[dict]: ...
    def batch_search(self, queries: torch.Tensor, top_k: int = 5) -> list[list[dict]]: ...
```

**Requirements:**

### `__init__`
- `embeddings` is shape `(N, D)` — N items, D-dimensional embeddings
- `ids` is a list of N string identifiers
- Store L2-normalized embeddings internally for efficient cosine similarity

### `search`
- `query` is shape `(D,)` — single query embedding
- L2-normalize the query
- Compute cosine similarity against all stored embeddings (dot product of normalized vectors)
- Return top-k results as `list[dict]`, each dict has `'id'` (str) and `'score'` (float)
- Results sorted by score descending

### `batch_search`
- `queries` is shape `(Q, D)` — Q query embeddings
- Vectorized: compute all similarities in one matmul
- Return `list[list[dict]]` — outer list has Q elements, each is a top-k result list
- Same dict format as `search`

In [ ]:
# YOUR CODE HERE

class EmbeddingSearcher:
    """Cosine similarity search over a fixed embedding database."""

    def __init__(self, embeddings: torch.Tensor, ids: list[str]) -> None:
        """Initialize with embeddings and their IDs.

        Args:
            embeddings: Embedding matrix of shape (N, D).
            ids: List of N string identifiers.
        """
        raise NotImplementedError

    def search(self, query: torch.Tensor, top_k: int = 5) -> list[dict]:
        """Find top-k most similar embeddings to a single query.

        Args:
            query: Query embedding of shape (D,).
            top_k: Number of results to return.

        Returns:
            List of dicts with 'id' and 'score', sorted by score descending.
        """
        raise NotImplementedError

    def batch_search(self, queries: torch.Tensor, top_k: int = 5) -> list[list[dict]]:
        """Vectorized search for multiple queries.

        Args:
            queries: Query embeddings of shape (Q, D).
            top_k: Number of results per query.

        Returns:
            List of Q result lists, each containing top-k dicts.
        """
        raise NotImplementedError

In [ ]:
# === TESTS — DO NOT MODIFY ===

torch.manual_seed(0)

# Build a known embedding database
N, D = 100, 64
embeddings = torch.randn(N, D)
ids = [f"item_{i:03d}" for i in range(N)]

searcher = EmbeddingSearcher(embeddings, ids)

# --- Test 1: Self-search should return score ~1.0 ---
query = embeddings[0]  # search for the first item
results = searcher.search(query, top_k=5)
assert len(results) == 5, f"Expected 5 results, got {len(results)}"
assert results[0]["id"] == "item_000", f"Self-search should return self first, got {results[0]['id']}"
assert abs(results[0]["score"] - 1.0) < 1e-4, (
    f"Self-similarity should be ~1.0, got {results[0]['score']:.6f}"
)

# --- Test 2: Ranking correctness with controlled embeddings ---
# Create embeddings where item_001 is closer to item_000 than item_002
controlled_embs = torch.zeros(3, 8)
controlled_embs[0] = torch.tensor([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
controlled_embs[1] = torch.tensor([0.9, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])  # closer to 0
controlled_embs[2] = torch.tensor([0.1, 0.9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])  # farther from 0
controlled_ids = ["close", "medium", "far"]

controlled_searcher = EmbeddingSearcher(controlled_embs, controlled_ids)
controlled_results = controlled_searcher.search(controlled_embs[0], top_k=3)

assert controlled_results[0]["id"] == "close", f"Expected 'close' first, got {controlled_results[0]['id']}"
assert controlled_results[1]["id"] == "medium", f"Expected 'medium' second, got {controlled_results[1]['id']}"
assert controlled_results[2]["id"] == "far", f"Expected 'far' third, got {controlled_results[2]['id']}"
assert controlled_results[0]["score"] > controlled_results[1]["score"] > controlled_results[2]["score"], (
    "Scores should be monotonically decreasing"
)

# --- Test 3: Batch search returns correct structure ---
queries = embeddings[:3]  # first 3 items as queries
batch_results = searcher.batch_search(queries, top_k=5)
assert len(batch_results) == 3, f"Expected 3 query results, got {len(batch_results)}"
for i, qr in enumerate(batch_results):
    assert len(qr) == 5, f"Query {i}: expected 5 results, got {len(qr)}"
    assert qr[0]["id"] == f"item_{i:03d}", (
        f"Query {i}: self-search should return self first, got {qr[0]['id']}"
    )
    assert abs(qr[0]["score"] - 1.0) < 1e-4, (
        f"Query {i}: self-similarity should be ~1.0, got {qr[0]['score']:.6f}"
    )

# --- Test 4: top_k clamping ---
small_results = searcher.search(embeddings[0], top_k=1)
assert len(small_results) == 1, f"top_k=1 should return 1 result, got {len(small_results)}"

print(f"Single search top result: {results[0]}")
print(f"Batch search shape: {len(batch_results)} queries x {len(batch_results[0])} results")
print("Problem 3 passed.")

---

# End of Exam

**Stop your timer.** Record your time and which problems you completed.

## Self-Assessment

| Problem | Completed? | Time Spent | Confidence (1-5) | Notes |
|---------|-----------|------------|-------------------|-------|
| Warmup 1: Sinusoidal Embedding | | | | |
| Warmup 2: Alpha Bar | | | | |
| Warmup 3: Gather/Reshape | | | | |
| Main 1: Multi-Head Attention | | | | |
| Main 2: Full DDPM | | | | |
| Main 3: Embedding Search | | | | |

**Review priority:** Any problem scored <= 3 on confidence needs a re-drill within 48 hours.